In [0]:
%run ../../utils/config.py

In [0]:
from pyspark.sql import functions as F

# Läs rådata från volymen
df_raw = spark.read.csv(
    RAW_DATA_PATH,
    header=True,
    inferSchema=True,
)

print(f"Rader inlästa: {df_raw.count():,}")
df_raw.printSchema()

# Lägg till metadata-kolumner
# Spåra när och varifrån data laddades in
df_bronze = df_raw \
    .withColumn("_ingested_at", F.current_timestamp()) \
    .withColumn("_source_file", F.lit(RAW_DATA_PATH))

# Skriv till bronze Delta-tabell
# Overwrite gör att man kan köra om scriptet utan problem
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.columnMapping.mode", "name") \
    .saveAsTable(BRONZE_TABLE)

print(f"Bronze-tabell skapad: {BRONZE_TABLE}")
spark.sql(f"SELECT count(*) as antal_rader FROM {BRONZE_TABLE}").display()